# Submits 14 y 15 - Ensamble z005 con BAJA+1 + BAJA+2 juntas

- **Target:** 1 = BAJA+1 o BAJA+2 (mejora confirmada en junio, z009: +22 M, y en mayo, z019: +18 a +22 M).
- **Modelos:** los 10 trials de `0_exploracion/z005_top10_final.csv` × 5 semillas = **50 modelos**.
- Train 202103-202106, predice 202108, promedio de probabilidades por cliente.
- Cortes: **14 → top 13.000**, **15 → top 12.000** (se manda último).
- Cada predicción se guarda en `probs_juntar/` (si se corta, al volver a correr sigue desde ahí).

Este notebook solo genera los submits.

In [1]:
# --- 1. Librerías y parámetros ---
import os
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

SEMILLAS    = [501419, 623353, 786349, 834569, 946331]
MESES_FINAL = [202103, 202104, 202105, 202106]
MES_PRED    = 202108
ARCHIVOS    = [(14, 13_000), (15, 12_000)]   # (número de submit, N)
CARPETA     = "probs_juntar"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}
elegidos = pd.read_csv("../0_exploracion/z005_top10_final.csv", sep=";", decimal=",",
                       encoding="utf-8-sig", index_col="trial")
elegidos

,ganancia_max_M_mean,ganancia_max_M_std,params_num_leaves,params_min_sum_hessian_in_leaf,user_attrs_best_iter
trial,,,,,
34,399.6,2.3,137.0,9.755574,538.0
36,395.5,4.4,61.0,7.745374,727.0
27,394.6,3.8,241.0,5.632060,578.0
29,394.5,2.0,193.0,5.968488,567.0
37,393.6,3.2,240.0,0.003039,888.0
31,392.8,2.3,203.0,1.444970,532.0
20,392.7,2.9,159.0,0.021543,843.0
9,392.5,2.5,65.0,1.250592,684.0
28,392.2,4.9,237.0,0.095672,826.0


In [2]:
# --- 2. Cargar datos, target (BAJA+1 + BAJA+2) y lags ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria2"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria2"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
final  = df[df["foto_mes"].isin(MESES_FINAL)]
futuro = df[df["foto_mes"] == MES_PRED][["numero_de_cliente"]].copy()
X_futuro = df.loc[futuro.index, variables]
datos_final = lgb.Dataset(final[variables], label=final["clase_binaria2"], free_raw_data=False)
print("Variables:", len(variables), "| positivos en train:", final["clase_binaria2"].sum())

Variables: 761 | positivos en train: 8067


In [3]:
# --- 3. Entrenar 10 trials × 5 semillas (retoma si se cortó) ---
for trial, r in elegidos.iterrows():
    p = {**params_fijos, "num_leaves": int(r["params_num_leaves"]),
         "min_sum_hessian_in_leaf": float(r["params_min_sum_hessian_in_leaf"])}
    for semilla in SEMILLAS:
        archivo = f"{CARPETA}/t{trial}_s{semilla}.npy"
        if os.path.exists(archivo):
            continue
        inicio = time.time()
        modelo = lgb.train({**p, "seed": semilla}, datos_final, num_boost_round=int(r["user_attrs_best_iter"]))
        np.save(archivo, modelo.predict(X_futuro))
        print(f"Trial {trial}, semilla {semilla}: {(time.time() - inicio) / 60:.1f} min", flush=True)

Trial 34, semilla 501419: 2.0 min


Trial 34, semilla 623353: 1.8 min


Trial 34, semilla 786349: 1.8 min


Trial 34, semilla 834569: 1.9 min


Trial 34, semilla 946331: 1.8 min


...

Trial 39, semilla 501419: 2.1 min


Trial 39, semilla 623353: 2.1 min


Trial 39, semilla 786349: 2.1 min


Trial 39, semilla 834569: 2.1 min


Trial 39, semilla 946331: 2.1 min


In [4]:
# --- 4. Ensamble y archivos para el bot ---
probs = [np.load(f"{CARPETA}/t{t}_s{s}.npy") for t in elegidos.index for s in SEMILLAS]
futuro["prob_ensamble"] = np.mean(probs, axis=0)
n_modelos = len(probs)
futuro.to_parquet(f"14-15_ens{n_modelos}_juntar_probs.parquet")

ordenados = futuro.sort_values("prob_ensamble", ascending=False)
for num, n in ARCHIVOS:
    archivo = f"{num}_ens{n_modelos}juntar_{n}.csv"
    ordenados.head(n)["numero_de_cliente"].astype("int64").to_csv(archivo, index=False, header=False)
    with open(archivo) as f:
        contenido = f.read()
    print(archivo, "| filas:", len(contenido.splitlines()), "| ¿e+?", "e+" in contenido)

# ¿Cuánto cambia respecto del submit 12 (mismo modelo, solo BAJA+2)?
previo = pd.read_parquet("final_ens100_probs.parquet")
top_prev = set(previo.nlargest(12_000, "prob_ensamble")["numero_de_cliente"])
top_new  = set(futuro.nlargest(12_000, "prob_ensamble")["numero_de_cliente"])
print("Coincidencia con el submit 12 (top 12.000):", round(len(top_prev & top_new) / 12_000 * 100, 1), "%")

14_ens50juntar_13000.csv | filas: 13000 | ¿e+? False
15_ens50juntar_12000.csv | filas: 12000 | ¿e+? False


Coincidencia con el submit 12 (top 12.000): 91.8 %


In [5]:
# --- 5. Anotar en resultados_experimentos.csv ---
ruta_res = "../0_exploracion/resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
for num, n in ARCHIVOS:
    desc = f"{num}_Ens{n_modelos}juntar_{n}"
    res = res[~res["descripcion"].str.startswith(desc)]
    res = pd.concat([res, pd.DataFrame([{
        "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
        "descripcion": f"{desc}: 10 trials z005 x {len(SEMILLAS)} semillas, target BAJA+1 + BAJA+2, lags 1 y 2 (submit)",
        "meses_train": str(MESES_FINAL), "mes_valid": MES_PRED, "semilla": str(SEMILLAS),
        "params": str(params_fijos) + " | trials " + str(list(elegidos.index)), "n_enviados": n}])], ignore_index=True)
res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(2)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
68,2026-10-08 14:22,14_Ens50juntar_13000: 10 trials z005 x 5 semil...,"[202103, 202104, 202105, 202106]",202108,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.005...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13000,NaN,NaN,NaN
69,2026-10-08 14:22,15_Ens50juntar_12000: 10 trials z005 x 5 semil...,"[202103, 202104, 202105, 202106]",202108,"[501419, 623353, 786349, 834569, 946331]","{'objective': 'binary', 'learning_rate': 0.005...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12000,NaN,NaN,NaN
